In [ ]:


import pandas as pd
from pathlib import Path
import plotly.express as px
import sys
sys.path.append(str(Path.home() / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen' / 'config'))
import plot as pt


EXPORT=True
PATH_PLOTS = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring")


indicator = 'MTIP_1'
plot_name = 'mtip_programmed_sunburst'


df_mtip = pd.read_excel(PATH_PLOTS/'Data'/'MTIP_2025_Projects.xlsx', sheet_name='export')
df_mtip.columns = [str(col) for col in df_mtip.columns]
df_mtip = df_mtip[df_mtip[' Status'].isin(['In Progress - Programmed', 'Programmed'])]


def remove_pre_hyphen(x, exp=' - '):
    if isinstance(x, str):
        try:
            x = x.split(exp, 1)[1]
        except Exception as e:
            pass
            print('How exceptional', e)
    return x

def remove_post_hyphen(x, exp=' - '):
    if isinstance(x, str):
        try:
            x = x.split(exp, 1)[0]
        except Exception as e:
            pass
            print('How exceptional!', e)
    return x


df_mtip['Overall Category'] = df_mtip[' Type'].apply(remove_post_hyphen)
df_mtip['Type'] = df_mtip[' Type'].apply(remove_pre_hyphen)
df_mtip = df_mtip[['Overall Category', 'Type', '2025']]
df_mtip = df_mtip.groupby(['Overall Category', 'Type'], as_index=False).agg(Cost=('2025', 'sum'))
df_mtip['Donut Hole'] = '2025 MTIP<br>Projects'
df_mtip.loc[df_mtip['Overall Category'].str.contains('Hwy'), 'Overall Category'] = 'Highways'


fig = px.sunburst(df_mtip
                    , path=['Donut Hole', 'Overall Category', 'Type']
                    , values='Cost')
                    # , hover_data=['Type', 'Cost'])


title = '<b>MTIP Planned Projects, 2025</b>  <br><sup>6-County Sacramento Region</sup> <br><sup><span style = "font-size:0.8em;">(Click inner circle to view more detail)</span></sup>'
# fig.update_traces(hovertemplate='%{customdata[0]}<br>%{customdata[1]:$,.0f}'.replace('(?)', ''))

fig.update_traces(hovertemplate="%{label}<br>%{value:$,.0f}<extra></extra>")


pt.plot_agol(fig, EXPORT, title, indicator, plot_name)


# df_qc = df_mtip.groupby(['Overall Category', 'Type'], as_index=False).agg(num_projects_2025=('2025', 'count'))
# df_qc.to_excel(PATH_PLOTS/'temp'/'Number of Planned MTIP Projects 2025.xlsx', index=False)
